# Creacion de la API DrChatPatin con POE

# Preparacion de poe

### Instalacion

In [ ]:
!pip install fastapi-poe

### Preparacion de la API KEY

In [ ]:
# from google.colab import userdata
# POE_KEY = userdata.get('POE_API_KEY')
POE_KEY = "wkyPK_lCauPp0psjBeyZWWwqT69_mie1Q4zOaAgA0rQ"

In [ ]:
query = """
Case summary
• Male, 27 y, previously healthy.
• Acute onset: high fever, vomiting, severe headache, rapid deterioration.
• Within hours: widespread purpuric skin lesions, cool extremities, somnolence, arterial hypotension unresponsive to fluids.
• Recent contact with a febrile child (no confirmed diagnosis).
Emergency department findings
• BP 78/42 mmHg, HR 132 bpm, RR 28/min, Temp 39.5 °C, SpO₂ 91 % on reservoir mask.
• Glasgow 12 / 15. Capillary refill > 4 s. Urine < 0.3 mL kg⁻¹ h⁻¹.
• Skin: distal coldness, petechiae, confluent purpura.
Initial labs
• WBC 2 100 / mm³ (relative neutropenia), Platelets 32 000 / mm³.
• INR 2.3, fibrinogen 89 mg/dL.
• Lactate 6.5 mmol/L, creatinine 2.4 mg/dL.
• Na⁺ 128 mmol/L, K⁺ 5.6 mmol/L, glucose 42 mg/dL.
• CRP 235 mg/L, procalcitonin 48.1 ng/mL.
• Basal serum cortisol 2.1 µg/dL.
• Adrenal CT: bilateral, macronodular hemorrhagic appearance.
• Blood cultures drawn.
Early management (first hour)
• Balanced crystalloid 30 mL kg⁻¹ (bolus 500 mL every 20 min).
• Empiric IV antibiotics (third-generation cephalosporin + glycopeptide).
• IV hydrocortisone 100 mg q8 h.
• Rapid glucose correction.
• Platelet & FFP transfusion.
• Transfer to ICU.
ICU day 1 summary
• Intubated for refractory shock and declining sensorium.
• Mechanical ventilation: VT 6 mL kg⁻¹, PEEP 7 cmH₂O, FiO₂ 0.6.
• Vasopressors: norepinephrine up to 0.5 µg kg⁻¹ min⁻¹; vasopressin 0.03 U min⁻¹.
• Continuous hydrocortisone infusion.
• Ongoing cryoprecipitate/FFP/platelets for coagulopathy.
• Invasive monitoring (arterial line, central venous line, capnography).
• Bedside echo: preserved LVEF, low preload.
Clinical course (days 2-4) — progressive fall in lactate, rising platelets & fibrinogen, vasopressor withdrawal, extubation day 4, neurologically intact.
"""

In [ ]:
import fastapi_poe as fp
import nest_asyncio
nest_asyncio.apply()

message = fp.ProtocolMessage(role="user", content=query)
ans_1 = ""

for partial in fp.get_bot_response_sync(messages=[message], bot_name="DrChatPatin-20B", api_key=POE_KEY):
    ans_1 += partial.text

print(ans_1)
print(100*"=")

message = fp.ProtocolMessage(role="user", content=f"Process the nex information about a diagnosis, analize again the data. Do a second thougth considering the following infomation, and then, provide a diferential diagnosis: \n{ans_1}")
ans_2 = ""
for partial in fp.get_bot_response_sync(messages=[message], bot_name="DrChatPatin-20B", api_key=POE_KEY):
    ans_2 += partial.text
print(ans_2)

# Implementacion del RAG

### Carga de la base de datos

In [ ]:
from langchain.document_loaders import DirectoryLoader
import pprint
path = "./NORD_DB"
global_pattern = '*.txt'
loader = DirectoryLoader(path=path, glob=global_pattern)
docs = loader.load()


print(f"loaded {len(docs)} documents")

### Descarga y preparacion del embedding

In [ ]:
# pip install \
#     torch==2.1.2 \
#     vllm==0.6.1 \
#     flash-attn==2.5.8 \
#     transformers==4.39.1 \
#     sentence-transformers==2.7.0 \
#     accelerate==0.27.2 \
#     peft==0.9.0

In [ ]:
import torch
from sentence_transformers import SentenceTransformer


N_GPU = torch.cuda.device_count()
DEVICE = torch.device(f'cuda:{N_GPU}' if torch.cuda.is_available() else 'cpu')


# model_name = "BAAI/bge-large-en-v1.5"
model_name = "BAAI/bge-m3"
encoder = SentenceTransformer(model_name, device='cpu')


EMBEDDING_DIM = encoder.get_sentence_embedding_dimension()
MAX_SEQ_LENGTH_IN_TOKENS = encoder.get_max_seq_length()


print(f"model_name: {model_name}")
print(f"EMBEDDING_DIM: {EMBEDDING_DIM}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH_IN_TOKENS}")


### Descomposicion de la base de datos en chunks

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import numpy as np
import time

CHUNK_SIZE = 768
CHUNK_OVERLAP = np.round(CHUNK_SIZE * .10, 0)
print(f"Chunk size: {CHUNK_SIZE}, Chunk overlap: {CHUNK_OVERLAP}")

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP)

chunks = child_splitter.split_documents(docs)
print(f"{len(docs)} docs split into {len(chunks)} child documents")

print(f"Working with embedding and creating the database for Milvus...")
start_time = time.time()

list_of_strings = [doc.page_content for doc in chunks if hasattr(doc, 'page_content')]

embeddings = torch.tensor(encoder.encode(list_of_strings))

embeddings = np.array(embeddings / np.linalg.norm(embeddings))

converted_values = list(map(np.float32, embeddings))

end_time = time.time()
print(f"Done!. Completed in {(end_time - start_time) / 60} min.")

dict_list = []

print("Processing chunks...")
start_time = time.time()

for chunk, vector in zip(chunks, converted_values):
    chunk_dict = {
        'chunk': chunk.page_content,
        'source': chunk.metadata.get('source',""),
        'vector': vector,
    }
    dict_list.append(chunk_dict)

end_time = time.time()
print(f"Chunks done!. Process completed in {end_time - start_time} seconds.")

### Integracion de los chunks y el embedding con milvus

In [ ]:
from pymilvus import MilvusClient
import time

# Configuración inicial
mc = MilvusClient("NORD_DATABASE.db")
COLLECTION_NAME = "RareDisseases"
EMBEDDING_DIM = 768  # Ajusta según tu modelo de embeddings

# Verificar si la colección ya existe
if not mc.has_collection(COLLECTION_NAME):
    # Crear colección solo si no existe
    mc.create_collection(
        COLLECTION_NAME,
        EMBEDDING_DIM,
        consistency_level="Eventually",
        auto_id=True,
        overwrite=False  # Importante: False para no borrar si existe
    )

    print("Start inserting entities")
    start_time = time.time()

    # Asume que 'dict_list' ya está generada con tus datos
    mc.insert(
        COLLECTION_NAME,
        data=dict_list,
        progress_bar=True
    )

    end_time = time.time()
    print(f"Milvus insert time for {len(dict_list)} vectors: {round(end_time - start_time, 2)} seconds")
else:
    print(f"La colección '{COLLECTION_NAME}' ya existe. Cargando desde disco...")
    # Opcional: Cargar la colección a memoria para optimizar búsquedas
    mc.load_collection(COLLECTION_NAME)

# Ahora puedes usar 'mc' para búsquedas RAG sin reprocesar

# Creacion de la API-V1

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import fastapi_poe as fp
import nest_asyncio
nest_asyncio.apply()

app = FastAPI(title="DrChatPatin")

class QueryRequest(BaseModel):
    query: str


@app.post("/generate/v1")
async def generate_response(request: QueryRequest):
    try:
        prompt = f"Conversation history and question: {request.query} \nRespuesta:"
        output = fp.ProtocolMessage(role="user", content=request.query)
        ans = ""

        for partial in fp.get_bot_response_sync(
            messages=[output],
            bot_name="DrChatPatin-20B",
            api_key=POE_KEY
        ):
            ans += partial.text
        return {"answer": ans}

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# @app.post("/generate/rag")
# async def generate_response_rag(request: QueryRequest):
#     try:

#         query_embeddings = torch.tensor(encoder.encode([request.prompt]))
#         query_embeddings = F.normalize(query_embeddings, p=2, dim=1)
#         query_embeddings = list(map(np.float32, query_embeddings))
#         OUTPUT_FIELDS = list(dict_list[0].keys())
#         OUTPUT_FIELDS.remove('vector')

#         results = mc.search(
#             COLLECTION_NAME,
#             data=query_embeddings,
#             output_fields=OUTPUT_FIELDS,
#             limit=10,
#             consistency_level="Eventually")

#         cites_set = set()

#         for result in results[0]:
#             source = result['entity']['source']
#             clean_source = source.replace("NORD_DB/", "").replace(".txt", "")
#             cites_set.add(clean_source)

#         cites = ", ".join(sorted(cites_set))


#         prompt = f"Instructions: {instructions} \nContext:{results[0]} \nQuery: {request.prompt}\nRespuesta:"
#         output = model.generate(prompt, sampling_params)
#         answer = output[0].outputs[0].text
#         response = f""" {answer.strip()}


#         **Diseases cited:** {cites}
#         """
#         return  response

#     except Exception as e:
#         raise HTTPException(status_code=500, detail=str(e))

## Prueba de la API

In [ ]:
import pprint

request = QueryRequest(query = query)
response = await generate_response(request)
pprint.pprint(response)

## Despliegue del servidor local

In [ ]:
import nest_asyncio
import uvicorn

# uvicorn.run(app, host="0.0.0.0", port=8000)
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)

### Prueba con consola de linux

In [ ]:
curl -X POST http://localhost:8000/generate/v1 \
     -H "Content-Type: application/json" \
     -d '{"query": "¿The patient reports a gradual onset of difficulty swallowing, initially with solids and progressively involving liquids. Episodes of chest discomfort are noted, ranging from mild to severe. Regurgitation of saliva and undigested food is frequent, particularly when lying down, and is sometimes accompanied by nighttime coughing. There are signs of unintentional weight loss due to reduced food intake. The patient also experiences dryness in the mouth and eyes. Episodes of aspiration have occurred, increasing the risk of respiratory infections."}'

### Prueba con PowerShell

In [ ]:
curl -Method POST "http://localhost:8000/generate/v1" `
     -ContentType "application/json" `
     -Body "{""query"":""The patient reports a gradual onset of difficulty swallowing, initially with solids and progressively involving liquids. Episodes of chest discomfort are noted, ranging from mild to severe. Regurgitation of saliva and undigested food is frequent, particularly when lying down, and is sometimes accompanied by nighttime coughing. There are signs of unintentional weight loss due to reduced food intake. The patient also experiences dryness in the mouth and eyes. Episodes of aspiration have occurred, increasing the risk of respiratory infections.""}"


In [ ]:
($resp = curl -Method POST "http://localhost:8000/generate/v1" `
     -ContentType "application/json" `
     -Body "{""query"":""The patient reports a gradual onset of difficulty swallowing, initially with solids and progressively involving liquids. Episodes of chest discomfort are noted, ranging from mild to severe. Regurgitation of saliva and undigested food is frequent, particularly when lying down, and is sometimes accompanied by nighttime coughing. There are signs of unintentional weight loss due to reduced food intake. The patient also experiences dryness in the mouth and eyes. Episodes of aspiration have occurred, increasing the risk of respiratory infections.""}").Content


## Port Forwarding

In [ ]:
from pyngrok import ngrok
from google.colab import userdata
# NGROK_API_KEY = userdata.get('POE_API_KEY')
NGROK_API_KEY = "2mDBvpwpYpLeG8rLJHEfpniRqND_5e29qSKAdtMeUsBbEShAQ"

# Terminate open tunnels if exist
ngrok.kill()

ngrok.set_auth_token(NGROK_API_KEY)
# Open an HTTPs tunnel on port 5000 for http://localhost:5000
ngrok_tunnel = ngrok.connect(addr="8000", proto="http", bind_tls=True)
print("vLLM  UI:", ngrok_tunnel.public_url)

# Creacion de la API-V2

In [ ]:
from fastapi import FastAPI
import nest_asyncio
import uvicorn

app = FastAPI()

@app.get('/test')
def testing():
    return "Test Complete"

uvicorn.run(app, host="0.0.0.0", port=8000)

# Otro 

In [1]:
import torch
if torch.cuda.is_available():
    print(f'Tarjeta grafica {torch.cuda.get_device_name(0)} disponible')
else:
    print('No hay ninguna tarjeta grafica para usar')

Tarjeta grafica NVIDIA GeForce RTX 3050 disponible


In [2]:
import ast
import torch
def load_corpus() -> list:
    with open('corpus.txt', 'r', encoding='utf8') as file:
        contenido_completo = file.read()

            # Encuentra el inicio de la lista (después de 'corpus = [')
        inicio_lista = contenido_completo.find('[')

        # Extrae solo la parte de la lista '[...]'
        string_de_la_lista = contenido_completo[inicio_lista:]

        # Convierte de forma segura el string a una lista de Python
        corpus = ast.literal_eval(string_de_la_lista)

    print(f"Load complete!!!, {len(corpus)} documents were processed")
    return corpus
print("Pre-loading the database")
corpus = load_corpus()

Pre-loading the database
Load complete!!!, 1249 documents were processed


In [ ]:
from sentence_transformers import SentenceTransformer

# Load the model
print("Loading model..")
model = SentenceTransformer("Qwen/Qwen3-Embedding-4B")
print("Model loaded and ready to use")

Loading model..


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
EMBEDDING_DIM = model.get_sentence_embedding_dimension()
MAX_SEQ_LENGTH_IN_TOKENS = model.get_max_seq_length()

print(f"Embedding dimensions: {EMBEDDING_DIM}")

In [4]:
from pymilvus import MilvusClient
c_name = "NRD.db"

client = MilvusClient(c_name)

if not client.has_collection(collection_name=c_name):
    client.create_collection(
        collection_name=c_name,
        dimension=EMBEDDING_DIM,  
    )



2025-09-17 07:51:47,781 [ERROR][create_connection]: Failed to create new connection using: NRD.db (_utils.py:48)


ModuleNotFoundError: No module named 'milvus_lite'

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
import numpy as np
import time

CHUNK_SIZE = 768
CHUNK_OVERLAP = np.round(CHUNK_SIZE * .10, 0)
print(f"Chunk size: {CHUNK_SIZE}, Chunk overlap: {CHUNK_OVERLAP}")

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP)

chunks = child_splitter.split_documents(corpus)
print(f"{len(corpus)} documents diseases split into {len(chunks)} child documents")

print(f"Working with embedding and creating the database for Milvus...")
start_time = time.time()

list_of_strings = [doc.page_content for doc in chunks if hasattr(doc, 'page_content')]

embeddings = torch.tensor(encoder.encode(list_of_strings))

embeddings = np.array(embeddings / np.linalg.norm(embeddings))

converted_values = list(map(np.float32, embeddings))

end_time = time.time()
print(f"Done!. Completed in {(end_time - start_time) / 60} min.")

dict_list = []

print("Processing chunks...")
start_time = time.time()

for chunk, vector in zip(chunks, converted_values):
    chunk_dict = {
        'chunk': chunk.page_content,
        'source': chunk.metadata.get('source',""),
        'vector': vector,
    }
    dict_list.append(chunk_dict)

end_time = time.time()
print(f"Chunks done!. Process completed in {end_time - start_time} seconds.")

In [ ]:
print("Preparing the database to the model")
corpus_embeddings = model.encode_document(corpus)
print("Database succefully prepared")

In [ ]:
print("Preparing the query to the model")
query_embbeding = model.encode_query('Can you provide information about Acrodysostosis?')
print("Query succefully prepared")

similarity_scores = model.similarity(query_embbeding, corpus_embeddings)[0]
score, indices = torch.topk(similarity_scores, k = 5)

for score, idx in zip(scores, indices):
        print(f"(Score: {score:.4f})", corpus[idx])